# Module 3 — Experimental Data Analysis with Pandas, siibra, and Nilearn

This notebook-style script combines tabular experimental data with atlas
information and ends by visualizing region-wise values on a brain template.

Learning goals:
- load and inspect tabular data with Pandas
- summarize values across experimental conditions
- work with region-level data
- fetch atlas information with siibra
- project region-wise values onto an atlas map
- visualize the result with Nilearn

Note:
This is a draft teaching notebook. Some siibra details may need adjustment
depending on the installed version and the chosen parcellation.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import plotting
import siibra

# %matplotlib inline

## 1. Retrieve the labelled map for Julich Brain Atlas on MNI 152 reference space

In [ ]:
labelled_map = siibra.get_map(parcellation="julich 3.1", space="mni152", maptype="labelled")
labelled_map

In [ ]:
# download the map as a nifti
map_nii = labelled_map.fetch()
map_nii

## 2. Plot the atlas using nilearn

In [ ]:
# get the colormap to display the map
colormap = labelled_map.get_colormap(fill_uncolored=True)

In [ ]:
# plot interactively with nilarn
plotting.view_img(
    map_nii,
    title="Region-wise experimental values on atlas",
    cmap=colormap,
    symmetric_cmap=False
)

## 3. Synthetic data to mimic experiment results

Normally, there would be data you can read from a csv, txt, tsv and so on where recordings.



In [ ]:
# Random selection of regions from Julich Brain Atlas
np.random.seed(16)
regions = [labelled_map.regions[i] for i in np.random.randint(0, len(labelled_map.regions), 30)]
tasks = ["rest", "motor task 1", "motor task 2", "visual task 1", "visual task 2", "visual task 3", "auditory task"]
experimental_data = pd.DataFrame(
    [{"region": r, **{t: np.random.randint(0, 250) for t in tasks}} for r in regions]
).set_index('region')
experimental_data

## 4. Basic statistics using pandas



In [ ]:
experimental_data.describe()

## 5. Basic plotting from the dataframe



In [ ]:
experimental_data.plot(kind='bar', figsize=(15, 5), grid="y-axis")

In [ ]:
experimental_data['rest'].plot(kind='bar', grid="y-axis", figsize=(15, 5))

In [ ]:
experimental_data[
    [col for col in experimental_data.columns if "visual" in col]
].plot(kind='bar', figsize=(15, 5))

## 6. Match experimental values to atlas regions

We create a simple lookup table from region name to value.



In [ ]:
selected_task = "rest"

In [ ]:
value_lookup = dict(zip(experimental_data.index, experimental_data[selected_task]))
colored_img = labelled_map.colorize(value_lookup).fetch()
plotting.view_img(
    colored_img,
    title="Region-wise experimental values on atlas",
    cmap='jet',
    symmetric_cmap=False
)

In [ ]:
# Alternative view: glass brain
plotting.plot_glass_brain(
    colored_img,
    title="Glass brain view of region-wise values",
    display_mode="lyrz",
    cmap="jet",
    threshold=0,
)
plotting.show()

## 7. Query gene expression for the regions in the DataFrame and explore the data 

In [ ]:
[siibra.vocabularies.GENE_NAMES[int(i)]["symbol"] for i in np.random.randint(0, len(siibra.vocabularies.GENE_NAMES), 10)]

In [ ]:
genes = [
    'OR13C4',
    'ZNF66',
    'MARCKSL1',
    'TERT',
    'CORO6',
    'LINC00588',
    'MDH1B',
    'BRF1',
    'LTA',
    'RIPPLY3'
]
gene_exps = {}
for region in regions:
    try:
        gene_exps[region] = siibra.features.get(
            labelled_map.parcellation.get_region(region),
            siibra.features.molecular.GeneExpressions,
            gene=genes,
        )[0]
    except:
        continue

## 8. Plot gene expressions as box plots

In [ ]:
for i, gene_exp in enumerate(gene_exps.values()):
    gene_exp.plot()

## 9. Calculate variance of gene expression levels per gene and region

In [ ]:
gene_exps_var = pd.DataFrame(
    [
        {
            "region": r,
            **{
                gene: gene_exp.data[gene_exp.data['gene'] == gene]['level'].var()
                for gene in genes
            }
        }
        for r, gene_exp in gene_exps.items()
    ]
).set_index('region')
gene_exps_var

## 10. Compute correlations

In [ ]:
# helper function that is not necessary but useful.
# One would create such functions for repeated use
def correlate_series(x, y, method="spearman"):
    """
    Correlate two pandas Series with shared index labels.

    method: "pearson", "spearman", or "kendall"
    """
    df = pd.concat([x.rename("x"), y.rename("y")], axis=1).dropna()

    if len(df) < 3:
        print("Need at least 3 paired observations after alignment/dropna.")
        return None

    return df["x"].corr(df["y"], method=method)

In [ ]:
experimental_data.loc[gene_exps_var.index, :].shape

In [ ]:
experimental_data.loc[gene_exps_var.index, selected_task]

In [ ]:
r_vals = {}
for gene in genes:
    r_vals[gene] = correlate_series(
        experimental_data.loc[gene_exps_var.index, selected_task],
        gene_exps_var[gene],
        method="spearman",
        log_y=True
    )
correlation = pd.Series(r_vals)
correlation

In [ ]:
correlation_wo_nans = correlation.dropna()
colors = np.where(correlation_wo_nans < 0, 'red', 'blue')
correlation_wo_nans.plot.bar(
    color=colors,
    title=f'correlation of {selected_task} values and gene experession levels'
)
plt.xlabel('genes')
plt.ylabel("r value")

### Hands on

Select different tasks/genes and try. You can access the list via `print(siibra.vocabularies.GENE_NAMES)`